In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import roc_auc_score
from collections import Counter
from rdkit.Chem import AllChem, DataStructs
from rdkit import Chem
from sklearn import metrics
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import random
import os
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from rdkit.Chem import Draw
from IPython.display import display
from sklearn.manifold import TSNE
import umap

In [ ]:
def smiles_to_morgan_fp(smiles: str, n_bits: int = 1024, radius: int = 2) -> np.ndarray:
    """
    Converts a SMILES string to a Morgan fingerprint.
    """
    mol = Chem.MolFromSmiles(smiles)    
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

def show_random_molecules(smiles_list, N=5):
    """
    Given a list of smiles, returns N visualizations of molecules
    """
    sampled_smiles = random.sample(smiles_list, min(N, len(smiles_list)))
    mols = [Chem.MolFromSmiles(smi) for smi in sampled_smiles]
    img = Draw.MolsToGridImage(mols, molsPerRow=10, subImgSize=(200, 200), legends=sampled_smiles)
    display(img)

**REAL NEGATIVES**

In [ ]:
df = pd.read_csv("/home/acomajuncosa/Downloads/activities_CHEMBL1794345.tsv", sep='\t', low_memory=False)
df = df[~df['Smiles'].isna()]
print("No pvalues: " + str([i for i in df['pChEMBL Value'].tolist() if not np.isnan(i)]))
print("All are nM: " + str(set([i for i in df['Standard Units']])))

# Generate pChEMBLs
df['pChEMBL_calculated'] = [-np.log10(i * 1e-09) for i in df['Standard Value']]

# Get actives and inactives
actives = df[df['pChEMBL_calculated'] >= 7]['Smiles'].tolist()
# inactives = df[df['pChEMBL_calculated'] < 7]['Smiles'].tolist()
inactives = df[df['pChEMBL_calculated'] < 5]['Smiles'].tolist()
print("Actives: " + str(len(actives)))
print("Inactives: " + str(len(inactives)))

# Fix random seed
np.random.seed(42)

# Choose N actives and 1 * N inactives
N = 7000
selected_actives = np.random.choice(actives, N, replace=False).tolist()
selected_inactives = np.random.choice(inactives, 1 * N, replace=False).tolist()
print("Actives: " + str(len(selected_actives)))
print("Inactives: " + str(len(selected_inactives)))

# Get ECFPs
print("Calculating ECFPs...")
actives_smiles = list(selected_actives)
inactives_smiles = list(selected_inactives)
selected_actives = [smiles_to_morgan_fp(i) for i in selected_actives]
selected_inactives = [smiles_to_morgan_fp(i) for i in selected_inactives]

# Create matrices
X = np.array(selected_actives + selected_inactives)
Y = np.array([1]*len(selected_actives) + [0]*len(selected_inactives))
print("Matrix shapes:")
print(X.shape, Y.shape)

No pvalues: []
All are nM: {'nM'}
Actives: 7364
Inactives: 86848
Actives: 7000
Inactives: 7000
Calculating ECFPs...
Matrix shapes:
(14000, 1024) (14000,)
